# Tutorial 02 · Python 基础 II：集合、函数、对象与异常

原始资料：[Tutorial 02 Ver2.docx](source/Tutorial%2002%20Ver2.docx)。本笔记按讲义顺序整理，并用可运行示例澄清原文中的不准确表述。

**使用方式：** Python 3.8+，只用标准库；在 Jupyter 中重启内核后从上到下运行。代码不需要输入，预期异常均已捕获。模块演示只在系统临时目录写文件，结束后自动清理。每个主题按「概念 → 示例 → 结果解析 → 练习/答案」学习；综合练习串联本篇知识。


## 1. 四种内置集合如何选择

| 类型 | 常见写法 | 顺序与访问 | 可变性 | 重复元素 |
|---|---|---|---|---|
| `list` | `[1, 2]` | 有顺序；按整数索引/切片 | 可变 | 允许 |
| `tuple` | `(1, 2)` | 有顺序；按整数索引/切片 | 元组自身不可变 | 允许 |
| `set` | `{1, 2}`；空集 `set()` | 无序；成员测试，无下标 | 集合自身可变 | 自动去重 |
| `dict` | `{"name": "Amy"}` | 按键访问；保留键的插入顺序 | 可变 | 键唯一，值可以重复 |

**先纠正概念：** `set` 可以增删；元素须可哈希。`dict` 不是「不允许重复值」。字典顺序从 Python 3.7 起是语言保证；CPython 3.6 已有该实现行为，不能笼统说所有 3.6 字典都无序。`list` 的「有序」指元素有确定位置，不是禁止重新排序。

参考：[Python 数据结构](https://docs.python.org/3/tutorial/datastructures.html)、[字典类型](https://docs.python.org/3/library/stdtypes.html#mapping-types-dict)。


## 2. List：索引与切片

索引从 `0` 开始，`-1` 是最后一项。`items[start:stop:step]` 左闭右开，省略边界时使用默认边界；切片产生新列表。列表可以混合不同类型，但这不意味着不同类型都能直接比较排序。


In [1]:
fruit_list = ["apple", "cherry", "banana", "cherry"]
items = ["abc", 34, True, 40, "male"]
print("水果、长度和类型：", fruit_list, len(fruit_list), type(fruit_list).__name__)
print("索引 2、倒数第 2 项：", items[2], items[-2])
print("切片 [2:4]：", items[2:4])
print("每隔一项：", items[::2])
print("反向：", fruit_list[::-1])


水果、长度和类型： ['apple', 'cherry', 'banana', 'cherry'] 4 list
索引 2、倒数第 2 项： True 40
切片 [2:4]： [True, 40]
每隔一项： ['abc', True, 'male']
反向： ['cherry', 'banana', 'cherry', 'apple']


**结果解析：** `[2:4]` 只含位置 2、3，所以得到 `[True, 40]`；重复的 `"cherry"` 保留。`[::-1]` 返回反向副本，不修改原列表。

**先想后运行：** 将位置 1 改成 `36`，再把 `[2:3]` 替换成两个元素，长度如何变化？下面重现讲义操作；讲义输出仍写 `34` 是笔误，正确值应为 `36`。


In [2]:
items = ["abc", 34, True, 40, "male"]
items[1] = 36
items[2:3] = [False, "Hong Kong"]
items.append("Yellow")
items.insert(1, "April")
print(items)
print("当前长度：", len(items))


['abc', 'April', 36, False, 'Hong Kong', 40, 'male', 'Yellow']
当前长度： 8


**结果解析：** 输出为 `['abc', 'April', 36, False, 'Hong Kong', 40, 'male', 'Yellow']`，长度是 8。切片赋值可以改变长度；`append(x)` 加一个对象，`extend(xs)` 逐个加入可迭代对象中的元素。

删除时，`remove(value)` 按值删除第一个匹配项；`pop(index)` 按位置删除并返回；`del items[index]` 删除指定位置；`clear()` 清空已有列表。`del name` 删除变量绑定，不能等同于保证对象立即销毁。


In [3]:
items.extend([5.9, "Chau"])
items.remove("Hong Kong")
print("pop(0) 移除：", items.pop(0))
print("pop() 移除：", items.pop())
del items[0]
print("剩余元素：", items)

single = [1]
single.append([2, 3])
expanded = [1]
expanded.extend([2, 3])
print("append 列表：", single, "；extend 列表：", expanded)
expanded.clear()
print("clear 后：", expanded)
del expanded
try:
    print(expanded)
except NameError:
    print("del expanded 后，该变量名不再绑定对象")


pop(0) 移除： abc
pop() 移除： Chau
剩余元素： [36, False, 40, 'male', 'Yellow', 5.9]
append 列表： [1, [2, 3]] ；extend 列表： [1, 2, 3]
clear 后： []
del expanded 后，该变量名不再绑定对象


### 排序、连接与复制：区分三个层次

`sort()` 原地排序并返回 `None`，`sorted()` 返回新列表，`+` 连接出新列表。`alias = original` 只增加别名；`copy()` 和 `[:]` 只复制外层，称为**浅拷贝**。内嵌列表仍共享时，修改其内容会影响两边；标准库 `copy.deepcopy()` 可递归复制下面的普通嵌套列表。

预测最后两行：如果向原列表的第一行追加 `99`，浅拷贝与深拷贝分别看到什么？


In [4]:
from copy import deepcopy

fruits = ["apple", "cherry", "banana"]
print("新升序列表：", sorted(fruits))
result = fruits.sort(reverse=True)
print("原地降序：", fruits, "；sort 返回：", result)
print("copy + 切片：", fruits.copy() + fruits[:])

original = [[1, 2], [3, 4]]
alias = original
shallow = original.copy()
deep = deepcopy(original)
original[0].append(99)
print("alias 是原对象：", alias is original)
print("浅拷贝外层独立、内层共享：", shallow is not original, shallow[0] is original[0])
print("原列表 / 浅拷贝：", original, shallow)
print("深拷贝：", deep)


新升序列表： ['apple', 'banana', 'cherry']
原地降序： ['cherry', 'banana', 'apple'] ；sort 返回： None
copy + 切片： ['cherry', 'banana', 'apple', 'cherry', 'banana', 'apple']
alias 是原对象： True
浅拷贝外层独立、内层共享： True True
原列表 / 浅拷贝： [[1, 2, 99], [3, 4]] [[1, 2, 99], [3, 4]]
深拷贝： [[1, 2], [3, 4]]


### 练习 1：列表推导式

把「创建空列表 → 循环 → 可选条件 → append」写成 `[表达式 for 元素 in 可迭代对象 if 条件]`。

1. 用循环和推导式分别得到 1～5 的平方。
2. 筛选 `apple, banana, cherry, kiwi, mango` 中含字母 `a` 的水果。
3. 将 `[45, 72, 90, 58, 86]` 中不低于 60 的成绩加 5 分。

下一格是参考答案。列表打印结果包含方括号和逗号，讲义的 `1 4 9 16 25` 只是数值示意。


In [5]:
squares_loop = []
for number in range(1, 6):
    squares_loop.append(number * number)
squares_comp = [number * number for number in range(1, 6)]
fruits = ["apple", "banana", "cherry", "kiwi", "mango"]
print("平方：", squares_comp, "；与循环相同：", squares_comp == squares_loop)
print("含 a：", [fruit for fruit in fruits if "a" in fruit])
print("及格成绩加分：", [score + 5 for score in [45, 72, 90, 58, 86] if score >= 60])


平方： [1, 4, 9, 16, 25] ；与循环相同： True
含 a： ['apple', 'banana', 'mango']
及格成绩加分： [77, 95, 91]


## 3. Tuple：不可变的是元组的槽位

元组支持索引、切片和连接，但不能重新给某个槽位赋值。单元素元组写成 `(42,)`，逗号不可省略。把元组转成列表、修改、再转回去，创建的是一个新元组。

**补充边界：** 元组可以包含可变对象。禁止 `record[1] = ...`，不代表禁止修改 `record[1]` 所引用列表的内容。参考：[元组与序列](https://docs.python.org/3/tutorial/datastructures.html#tuples-and-sequences)。


In [6]:
fruit_tuple = ("apple", "banana", "cherry", "apple", "cherry")
print("末项 / 切片：", fruit_tuple[-1], fruit_tuple[1:3])
try:
    fruit_tuple[0] = "kiwi"
except TypeError as error:
    print("不能修改元组槽位：", type(error).__name__)

record = ("Amy", [80, 90])
record[1].append(95)
print("内嵌列表仍可修改：", record)
print("单元素元组 / 普通括号：", type((42,)).__name__, type((42)).__name__)


末项 / 切片： cherry ('banana', 'cherry')
不能修改元组槽位： TypeError
内嵌列表仍可修改： ('Amy', [80, 90, 95])
单元素元组 / 普通括号： tuple int


**结果解析：** 直接赋值触发 `TypeError`，但 Amy 的成绩列表成功新增 `95`；不要把「元组不可变」理解成整个对象图都不可变。

**练习 2：** 保留 `fruit_tuple`，创建末项改成 `kiwi` 的新元组，再把 `(1, 2, 3)` 接到前面。判断原元组最后一项是否改变。


In [7]:
editable = list(fruit_tuple)
editable[-1] = "kiwi"
new_tuple = tuple(editable)
combined = (1, 2, 3) + new_tuple
print("原元组：", fruit_tuple)
print("新元组：", new_tuple)
print("连接后：", combined)


原元组： ('apple', 'banana', 'cherry', 'apple', 'cherry')
新元组： ('apple', 'banana', 'cherry', 'apple', 'kiwi')
连接后： (1, 2, 3, 'apple', 'banana', 'cherry', 'apple', 'kiwi')


## 4. Set：去重、成员测试与集合运算

集合自身可变，无位置索引，不保证迭代顺序。元素须可哈希，普通列表不能直接放入集合。`{}` 是空字典，空集合要写 `set()`。

`add(x)` 加一个元素；`update(xs)` 加多个元素。`remove(x)` 在缺失时抛出 `KeyError`，`discard(x)` 在缺失时不报错。下面打印同类字符串集合时用 `sorted()`，只是为了让输出可比较，集合本身仍然无序。


In [8]:
fruit_set = {"apple", "banana", "cherry", "apple"}
print("去重：", sorted(fruit_set), "；banana 是否存在：", "banana" in fruit_set)
fruit_set.add("mango")
fruit_set.update(["kiwi", "orange"])
fruit_set.remove("orange")
fruit_set.discard("not-present")
print("增删后：", sorted(fruit_set))
try:
    fruit_set.remove("not-present")
except KeyError:
    print("remove 缺失元素会触发 KeyError")
try:
    fruit_set.add(["pear"])
except TypeError:
    print("list 不可哈希，不能作为 set 元素")
empty_copy = fruit_set.copy()
empty_copy.clear()
print("清空副本：", empty_copy, "；原集合长度：", len(fruit_set))


去重： ['apple', 'banana', 'cherry'] ；banana 是否存在： True
增删后： ['apple', 'banana', 'cherry', 'kiwi', 'mango']
remove 缺失元素会触发 KeyError
list 不可哈希，不能作为 set 元素
清空副本： set() ；原集合长度： 5


### 练习 3：谁同时参加两门课？

设 Python 班为 `{Amy, Bo, Chen}`，统计班为 `{Bo, Dana}`，求并集、交集、差集、对称差集。

| 运算 | 含义 | 方法 / 运算符 |
|---|---|---|
| 并集 | 至少参加一门 | `union()` / `|` |
| 交集 | 两门都参加 | `intersection()` / `&` |
| 差集 | 在左边、但不在右边 | `difference()` / `-` |
| 对称差集 | 恰好参加一门 | `symmetric_difference()` / `^` |

`union()` 返回新集合；`update()` 修改原集合。方法可接受一般可迭代对象，`|` 的这两个操作数应为集合类对象。


In [9]:
python_class = {"Amy", "Bo", "Chen"}
statistics_class = {"Bo", "Dana"}
print("并集：", sorted(python_class | statistics_class))
print("交集：", sorted(python_class & statistics_class))
print("Python 独有：", sorted(python_class - statistics_class))
print("统计班独有：", sorted(statistics_class - python_class))
print("恰好一门：", sorted(python_class ^ statistics_class))
print("方法接受列表：", sorted(python_class.union(["Eve", "Amy"])))
print("原集合：", sorted(python_class))


并集： ['Amy', 'Bo', 'Chen', 'Dana']
交集： ['Bo']
Python 独有： ['Amy', 'Chen']
统计班独有： ['Dana']
恰好一门： ['Amy', 'Chen', 'Dana']
方法接受列表： ['Amy', 'Bo', 'Chen', 'Eve']
原集合： ['Amy', 'Bo', 'Chen']


## 5. Dict：键值映射与动态视图

字典的键须可哈希；重复写同一个键会覆盖值，不会增加第二个同名键；不同键可以具有相同值。`d[key]` 访问必需键，`d.get(key, default)` 适合可选键。

`keys()`、`values()`、`items()` 是随字典变化的**动态视图**，不是固定列表；`list(d.keys())` 才是当时的快照。`copy()` 对嵌套结构同样是浅拷贝。

讲义的 `del thisdict.["brand"]` 多了一个点，正确写法是 **`del thisdict["brand"]`**。参考：[字典视图](https://docs.python.org/3/library/stdtypes.html#dictionary-view-objects)。


In [10]:
car = {"brand": "Ford", "model": "Mustang", "year": 1964}
keys_view = car.keys()
keys_snapshot = list(car.keys())
car["color"] = "red"
car.update({"year": 2020})
print("取值 / 默认值：", car["brand"], car.get("owner", "未登记"))
print("动态视图：", list(keys_view), "；旧快照：", keys_snapshot)
car_copy = car.copy()
print("pop 返回值：", car.pop("color"))
del car["brand"]
print("删除后的动态视图：", list(keys_view))
print("values / items：", list(car.values()), list(car.items()))
print("先前的浅拷贝：", car_copy)
print("键重复时覆盖，值可重复：", {"Amy": 80, "Bo": 80, "Amy": 95})
car.clear()
print("clear 后视图：", list(keys_view))


取值 / 默认值： Ford 未登记
动态视图： ['brand', 'model', 'year', 'color'] ；旧快照： ['brand', 'model', 'year']
pop 返回值： red
删除后的动态视图： ['model', 'year']
values / items： ['Mustang', 2020] [('model', 'Mustang'), ('year', 2020)]
先前的浅拷贝： {'brand': 'Ford', 'model': 'Mustang', 'year': 2020, 'color': 'red'}
键重复时覆盖，值可重复： {'Amy': 95, 'Bo': 80}
clear 后视图： []


**结果解析：** `keys_view` 先看到新增的 `color`，之后又反映键被删除、字典被清空；`keys_snapshot` 一直保留最初三个键。`car_copy` 的外层独立，所以不受原字典的顶层增删影响。

**练习 4：** 用字典保存 `Amy: 80, Bo: 65, Chen: 90`，将 Bo 改成 70，添加 Dana 85，再用字典推导式选出至少 80 分的人。额外说明为什么嵌套字典/列表不能靠 `.copy()` 完全隔离。


In [11]:
scores_by_name = {"Amy": 80, "Bo": 65, "Chen": 90}
scores_by_name.update({"Bo": 70, "Dana": 85})
high_scores = {name: score for name, score in scores_by_name.items() if score >= 80}
print("全部成绩：", scores_by_name)
print("至少 80 分：", high_scores)

nested = {"Amy": [80, 90]}
nested_copy = nested.copy()
nested_copy["Amy"].append(100)
print("浅拷贝共享成绩列表：", nested)


全部成绩： {'Amy': 80, 'Bo': 70, 'Chen': 90, 'Dana': 85}
至少 80 分： {'Amy': 80, 'Chen': 90, 'Dana': 85}
浅拷贝共享成绩列表： {'Amy': [80, 90, 100]}


## 6. 函数：参数、返回值与不定长参数

定义中的名字叫**形参**，调用时传入的值叫**实参**。默认参数可省略；缺少必需参数会触发 `TypeError`。`return` 将结果交给调用者，`print` 只展示信息。

`*args` 收集额外的位置实参，得到元组；`**kwargs` 收集额外的关键字实参，得到字典。`args`、`kwargs` 是约定名称，星号才决定行为。


In [12]:
def full_name(first, last="Chau"):
    return f"{first} {last}"

def list_children(*kids):
    return type(kids).__name__, ", ".join(kids)

def describe_person(**details):
    return type(details).__name__, details

print(full_name("Amy"))
print(full_name(last="Chan", first="Bo"))
print(list_children("Emil", "Tobias", "Linus"))
print(describe_person(fname="Tobias", lname="Refsnes"))
try:
    full_name()
except TypeError:
    print("未提供 first：TypeError")


Amy Chau
Bo Chan
('tuple', 'Emil, Tobias, Linus')
('dict', {'fname': 'Tobias', 'lname': 'Refsnes'})
未提供 first：TypeError


### `/` 与 `*`：限制调用方式

讲义签名 `def calculate(a, b, /, s, *, c, d=9)` 中：

- `/` 左边的 `a, b` 只能按位置传入。
- 中间的 `s` 可以按位置或按关键字传入。
- 单独的 `*` 右边的 `c, d` 只能按关键字传入；`d` 的默认值是 9。

计算 `a + d - b - c` 时，传 `5, 6, 'S', c=7` 得到 `1`。函数签名中的单独 `*` 不创建一个 args 元组。详见 [特殊参数](https://docs.python.org/3/tutorial/controlflow.html#special-parameters)。


In [13]:
def calculate(a, b, /, s, *, c, d=9):
    print(f"{s} 是普通参数")
    return a + d - b - c

print("位置传 s：", calculate(5, 6, "S", c=7))
print("关键字传 s：", calculate(5, 6, s="S", c=7))
for description, bad_call in [
    ("a、b 不能写成关键字", lambda: calculate(a=5, b=6, s="S", c=7)),
    ("c 不能按位置传入", lambda: calculate(5, 6, "S", 7)),
]:
    try:
        bad_call()
    except TypeError:
        print(description, "→ TypeError")


S 是普通参数
位置传 s： 1
S 是普通参数
关键字传 s： 1
a、b 不能写成关键字 → TypeError
c 不能按位置传入 → TypeError


### 练习 5：把分数统计封装成函数

编写 `average_score(*scores, digits=1)`，接收任意数量的分数，返回保留指定小数位的平均数。没有分数时主动抛出 `ValueError`；`digits` 只能按关键字传入。

预测 `average_score(70, 80, 95)` 的结果。扩展思考：如果参数默认值是可变列表，会在多次调用间共享；需要新列表时用 `None` 默认值并在函数内部创建。


In [14]:
def average_score(*scores, digits=1):
    if not scores:
        raise ValueError("至少需要一个分数")
    return round(sum(scores) / len(scores), digits)

print("平均分：", average_score(70, 80, 95))
print("保留两位：", average_score(70, 80, 95, digits=2))
try:
    average_score()
except ValueError as error:
    print("空输入：", error)


平均分： 81.7
保留两位： 81.67
空输入： 至少需要一个分数


## 7. 类与对象：Person 和 Student

类描述对象的数据与行为；实例是按类创建的具体对象。`__init__` 初始化实例，`self` 指向当前实例，`__str__` 必须返回字符串供 `str()`、`print()` 使用。`self` 是约定名称，不是关键字。

讲义「类里的每个函数第一个参数都必须是 self」说得过宽：它适用于常规实例方法；静态方法没有自动传入的实例，类方法通常接收 `cls`。`Student(Person)` 建立继承，空类体用 `pass`。


In [15]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __str__(self):
        return f"{self.name}({self.age})"

    def greet(self):
        return f"Hello, my name is {self.name}"

class Student(Person):
    pass

person = Person("John", 36)
student = Student("Amy", 20)
print(person)
print(f"{person.name} with age {person.age}")
print(student, student.greet())
print("student 是 Person 的实例：", isinstance(student, Person))


John(36)
John with age 36
Amy(20) Hello, my name is Amy
student 是 Person 的实例： True


**结果解析：** `Student` 的类体虽然只有 `pass`，仍继承初始化方法、属性访问和 `greet()`，所以可以直接调用 `Student("Amy", 20)`。

**练习 6：** 定义 `EnrolledStudent(Student)`，增加 `student_id`，用 `super().__init__()` 复用姓名与年龄初始化，再覆盖 `__str__`。`super()` 在这里找到继承链中的父类实现。


In [16]:
class EnrolledStudent(Student):
    def __init__(self, name, age, student_id):
        super().__init__(name, age)
        self.student_id = student_id

    def __str__(self):
        return f"{self.student_id}: {self.name}({self.age})"

enrolled = EnrolledStudent("Bo", 21, "S002")
print(enrolled)
print("继承的方法：", enrolled.greet())


S002: Bo(21)
继承的方法： Hello, my name is Bo


## 8. 多态：统一调用 move()，得到不同动作

子类重新实现父类同名方法叫**覆盖（override）**。循环不必判断车辆类型，直接调用 `vehicle.move()`，运行时会选择对应实现。

**讲义的重要纠错：** 同一个 Python 类中先写 `def move(self)`，再写 `def move(self, height)`，后一个会替换前一个名字绑定，不能按实参数量自动选择；这不是传统的签名重载。原文第二个 `def` 还缺少冒号。下面先用可运行版本演示同名定义被替换。参考：[继承与方法覆盖](https://docs.python.org/3/tutorial/classes.html#inheritance)。


In [17]:
class PlaneWithDuplicateMethod:
    def move(self):
        return "Fly!"

    def move(self, height):
        return f"Move up with height {height} meters"

broken_plane = PlaneWithDuplicateMethod()
try:
    broken_plane.move()
except TypeError:
    print("后定义覆盖前定义：move() 缺少 height，触发 TypeError")
print("传入 height 才能调用：", broken_plane.move(1000))


后定义覆盖前定义：move() 缺少 height，触发 TypeError
传入 height 才能调用： Move up with height 1000 meters


**修正方式：** 只保留一个 `Plane.move(self, height=None)`，用默认参数同时支持 `move()` 与 `move(1000)`。这是一种函数内部的参数处理方式，不是声明了两个重载方法。

下面的 `Car` 直接继承，`Boat` 和 `Plane` 覆盖方法；它们仍继承车辆的 `brand`、`model` 初始化。先预测三次无参调用的结果，再运行检查。


In [18]:
class Vehicle:
    def __init__(self, brand, model):
        self.brand = brand
        self.model = model

    def move(self):
        return "Move!"

class Car(Vehicle):
    pass

class Boat(Vehicle):
    def move(self):
        return "Sail!"

class Plane(Vehicle):
    def move(self, height=None):
        if height is None:
            return "Fly!"
        return f"Move up with height {height} meters"

vehicles = [Car("Ford", "Mustang"), Boat("Ibiza", "Touring 20"), Plane("Boeing", "747")]
for vehicle in vehicles:
    print(vehicle.brand, vehicle.model, "→", vehicle.move())
print("带高度参数：", vehicles[-1].move(1000))


Ford Mustang → Move!
Ibiza Touring 20 → Sail!
Boeing 747 → Fly!
带高度参数： Move up with height 1000 meters


## 9. 模块：真的创建 .py 文件并导入

常见模块是 `.py` 文件，可保存函数、类、常量。`import module` 用模块名访问；`import module as alias` 使用别名；`from module import name` 直接绑定指定名字。这里不需要 pip 安装任何东西。

下面在 `TemporaryDirectory` 中创建独有名称的教学模块，临时加入 `sys.path`，演示三种导入后在 `finally` 中恢复路径与模块缓存。目录退出时自动删除，仓库不留下配套 `.py` 或缓存文件。


In [19]:
import importlib
import sys
import tempfile
from pathlib import Path

module_name = "_week3_tutorial_demo_9fd8"
original_path = sys.path.copy()
sentinel = object()
previous_module = sys.modules.get(module_name, sentinel)
module_text = (
    'def greeting(name):\n'
    '    return "Hello, " + name\n'
    '\n'
    'person1 = {"name": "John", "age": 36, "country": "Norway"}\n'
)

with tempfile.TemporaryDirectory(prefix="week3_module_") as temp_directory:
    module_file = Path(temp_directory) / f"{module_name}.py"
    module_file.write_text(module_text, encoding="utf-8")
    try:
        sys.modules.pop(module_name, None)
        sys.path.insert(0, temp_directory)
        importlib.invalidate_caches()
        import _week3_tutorial_demo_9fd8
        import _week3_tutorial_demo_9fd8 as mx
        from _week3_tutorial_demo_9fd8 import person1

        print("import：", _week3_tutorial_demo_9fd8.greeting("Jonathan"))
        print("as 别名：", mx.person1["age"])
        print("from 导入：", person1["country"])
        print("别名是否引用同一模块：", mx is _week3_tutorial_demo_9fd8)
    finally:
        sys.path[:] = original_path
        if previous_module is sentinel:
            sys.modules.pop(module_name, None)
        else:
            sys.modules[module_name] = previous_module
        sys.path_importer_cache.pop(temp_directory, None)
        importlib.invalidate_caches()

print("临时目录已清理：", not Path(temp_directory).exists())
print("sys.path 已恢复：", sys.path == original_path)


import： Hello, Jonathan
as 别名： 36
from 导入： Norway
别名是否引用同一模块： True
临时目录已清理： True
sys.path 已恢复： True


**结果解析：** 三种写法分别输出问候、`36`、`Norway`；模块别名引用同一模块。`from` 导入后用 `person1` 即可访问；它仍引用模块中的同一字典，并不是深拷贝。

### Class / Module / Package / Library 不是严格层级

| 名称 | 更准确的理解 | 例子 |
|---|---|---|
| Class | 对象类型的定义；模块不一定含类 | 本篇 `Person` |
| Module | 可被导入的代码单元；常见为 `.py`，也有内置/扩展模块 | `math`、临时模块 |
| Package | 可以包含子模块/子包的模块 | `xml`、`xml.etree` |
| Library | 对一组可复用功能的泛称，不是语法结构 | Python 标准库 |

普通包通常有 `__init__.py`；**命名空间包**可以没有该文件，并可横跨多个目录。安装发行包与 `import` 的模块/包名也不必相同，因此不能机械套用「Library → Package → Module → Class」。参考：[Python 导入系统与命名空间包](https://docs.python.org/3/reference/import.html#namespace-packages)。

**练习 7：** 将临时模块的 `person1` 导入名改成 `from ... import person1 as person_info`，访问方式应为 `person_info["age"]`。


## 10. 异常处理：try / except / else / finally

| 部分 | 何时运行 | 常见用途 |
|---|---|---|
| `try` | 先尝试执行 | 可能失败的转换或操作 |
| `except 指定异常` | `try` 抛出匹配异常时 | 给出恢复方案或错误说明 |
| `else` | `try` 正常结束、未触发异常 | 继续处理成功结果 |
| `finally` | 通常在退出整个语句前执行 | 释放资源或记录结束 |

优先捕获具体异常，避免讲义中的裸 `except:` 把中断等也吞掉。`finally` 不是任意情况下的绝对保证，例如进程被强行终止时不能依赖它。参考：[错误与异常](https://docs.python.org/3/tutorial/errors.html)。


In [20]:
def parse_integer(text):
    try:
        number = int(text)
    except ValueError:
        print(f"{text!r} 不是合法整数文本")
    else:
        print(f"转换成功：{number}，平方为 {number ** 2}")
    finally:
        print("本次转换结束")

for text in ["42", "hello"]:
    parse_integer(text)


转换成功：42，平方为 1764
本次转换结束
'hello' 不是合法整数文本
本次转换结束


### 主动 raise，并辨清 TypeError 与 ValueError

`TypeError` 表示输入类型不符合约定；`ValueError` 表示类型可接受但数值/内容无效。下面约定成绩必须是 **0～100 的内置整数**，布尔值也拒绝，因此使用 `type(score) is int`；这是本练习的业务约定，不代表所有函数都应如此严格。

讲义先 `print(H)` 再检查 `H` 类型：如果 H 尚未定义，第一行就抛出 `NameError`，根本到不了 `raise TypeError`。下面用独立空命名空间稳定复现，不受此前内核是否定义过 H 的影响。


In [21]:
try:
    exec("print(H)", {})
except NameError:
    print("未定义 H：先发生 NameError，后续类型检查不会执行")

def validate_score(score):
    if type(score) is not int:
        raise TypeError("成绩必须是内置整数，且不能是 bool")
    if not 0 <= score <= 100:
        raise ValueError("成绩必须在 0～100 之间")
    return score

for candidate in [80, "80", -1, 101, True]:
    try:
        valid_score = validate_score(candidate)
    except (TypeError, ValueError) as error:
        print(repr(candidate), "→", type(error).__name__, str(error))
    else:
        print(candidate, "→ 合法成绩", valid_score)


未定义 H：先发生 NameError，后续类型检查不会执行
80 → 合法成绩 80
'80' → TypeError 成绩必须是内置整数，且不能是 bool
-1 → ValueError 成绩必须在 0～100 之间
101 → ValueError 成绩必须在 0～100 之间
True → TypeError 成绩必须是内置整数，且不能是 bool


## 11. 综合练习：清洗成绩并生成班级报告

数据用 `list[dict]` 组织，每条记录含学号、姓名、分组、成绩；课程元信息用元组保存。

**任务：**

1. 使用 `validate_score()` 保留合法成绩，将错误记录及原因单独保存。
2. 用字典生成学号到成绩的映射，用集合找出分组和缺交同学。
3. 用推导式筛选至少 60 分的人，计算平均分并按成绩降序排列。
4. 将数据交给对象，生成每个人的结果字符串。

约定学号唯一；本数据里 Dana 是字符串成绩，Eve 超出范围，必须记录错误，不能悄悄丢失。下一格先给清洗参考答案，再完成报告。


In [22]:
course_info = ("DSC2001", "Tutorial 02")
raw_records = [
    {"student_id": "S001", "name": "Amy", "group": "A", "score": 82},
    {"student_id": "S002", "name": "Bo", "group": "A", "score": 58},
    {"student_id": "S003", "name": "Chen", "group": "B", "score": 91},
    {"student_id": "S004", "name": "Dana", "group": "B", "score": "85"},
    {"student_id": "S005", "name": "Eve", "group": "A", "score": 105},
    {"student_id": "S006", "name": "Finn", "group": "B", "score": 73},
]
clean_records = []
rejected_records = []
for record in raw_records:
    try:
        validate_score(record["score"])
    except (TypeError, ValueError) as error:
        rejected_records.append({"student_id": record["student_id"], "reason": str(error)})
    else:
        clean_records.append(record.copy())

print("课程：", course_info)
print("合法 / 错误记录数：", len(clean_records), len(rejected_records))
for rejected in rejected_records:
    print("待修正：", rejected)


课程： ('DSC2001', 'Tutorial 02')
合法 / 错误记录数： 4 2
待修正： {'student_id': 'S004', 'reason': '成绩必须是内置整数，且不能是 bool'}
待修正： {'student_id': 'S005', 'reason': '成绩必须在 0～100 之间'}


**结果解析：** 保留 4 条，拒绝 2 条；Dana 违反类型约定，Eve 违反范围约定。这里只复制记录的外层已经足够，因为字段值都是不可变的字符串或整数；若字段中再嵌套列表，要重新考虑复制深度。

**继续练习：** 名册为 `S001`～`S007`。分别计算「未提交任何记录」和「尚无有效成绩」的学号；这两个集合是否相同？尝试先写代码，再查看下一格答案。


In [23]:
registered_ids = {f"S{number:03d}" for number in range(1, 8)}
submitted_ids = {record["student_id"] for record in raw_records}
valid_ids = {record["student_id"] for record in clean_records}
score_lookup = {record["student_id"]: record["score"] for record in clean_records}
passing_names = [record["name"] for record in clean_records if record["score"] >= 60]
ranked_records = sorted(clean_records, key=lambda record: record["score"], reverse=True)

print("分组：", sorted({record["group"] for record in raw_records}))
print("学号 → 成绩：", score_lookup)
print("未提交：", sorted(registered_ids - submitted_ids))
print("尚无有效成绩：", sorted(registered_ids - valid_ids))
print("及格姓名：", passing_names)
print("有效成绩平均数：", average_score(*(record["score"] for record in clean_records)))
print("排名：", [(record["name"], record["score"]) for record in ranked_records])


分组： ['A', 'B']
学号 → 成绩： {'S001': 82, 'S002': 58, 'S003': 91, 'S006': 73}
未提交： ['S007']
尚无有效成绩： ['S004', 'S005', 'S007']
及格姓名： ['Amy', 'Chen', 'Finn']
有效成绩平均数： 76.0
排名： [('Chen', 91), ('Amy', 82), ('Finn', 73), ('Bo', 58)]


**结果解析：** 未提交只有 `S007`；尚无有效成绩是 `S004, S005, S007`。4 条有效成绩平均分为 `76.0`，及格者为 Amy、Chen、Finn；排名为 Chen 91、Amy 82、Finn 73、Bo 58。

**综合练习后半：** 按 A/B 组分别统计有效平均分，再定义 `GradeRecord`，用 `__str__()` 统一输出学号、姓名、成绩与及格状态。这里对象只接收经过清洗的记录。


In [24]:
group_averages = {}
for group in sorted({record["group"] for record in clean_records}):
    group_scores = [record["score"] for record in clean_records if record["group"] == group]
    group_averages[group] = average_score(*group_scores)
print("分组有效平均数：", group_averages)

class GradeRecord:
    def __init__(self, student_id, name, score):
        self.student_id = student_id
        self.name = name
        self.score = validate_score(score)

    def passed(self, *, threshold=60):
        return self.score >= threshold

    def __str__(self):
        status = "及格" if self.passed() else "未及格"
        return f"{self.student_id} {self.name}: {self.score} ({status})"

for record in ranked_records:
    grade = GradeRecord(record["student_id"], record["name"], record["score"])
    print(grade)


分组有效平均数： {'A': 70.0, 'B': 82.0}
S003 Chen: 91 (及格)
S001 Amy: 82 (及格)
S006 Finn: 73 (及格)
S002 Bo: 58 (未及格)


## 12. 复习清单与迁移练习

本例 A 组有效平均分为 `70.0`，B 组为 `82.0`。`GradeRecord` 用方法封装及格判断；同一份数据可以同时使用列表保存记录、字典查找成绩、集合比较名册、元组保存固定元信息。

- 能说明 `append` 与 `extend`、索引与切片、别名与浅拷贝的区别。
- 能解释为什么 set 可变、tuple 内的列表仍可变、字典只有键必须唯一。
- 能使用推导式与四种集合运算，并识别动态字典视图。
- 能读懂 `*args`、`**kwargs`、`/` 和 `*`，正确传入参数。
- 能用 Person/Student 说明继承，用 Vehicle/Boat/Plane 说明覆盖和多态。
- 能区分同名方法替换与传统签名重载，使用默认参数修正示例。
- 能创建模块并用三种方式导入，辨清模块、包、类与库。
- 能用具体异常、`else`、`finally` 和 `raise` 处理成功与失败路径。

**迁移练习：** 将 Dana 修正为整数 `85`、Eve 修正为 `95`，重新执行综合练习；应有 6 条有效记录，平均分约 `80.7`，尚无有效成绩只剩 `S007`。再将及格线改为 75，尝试调用 `grade.passed(threshold=75)`。

回到[原始讲义](source/Tutorial%2002%20Ver2.docx)，逐段检查自己能否解释代码为什么产生该结果，而不仅仅是记住写法。
